# Notebook de limpieza y extracción de atributos - familia válvulas
## Proyecto: ARGOS – Sistema Inteligente de Clasificación y Localización de Materiales

# TRABAJO CONTINUA EN DESARROLLO...

### Llamado de dependencias

In [45]:
import pandas as pd
import re, unicodedata
from fractions import Fraction
from pathlib import Path

### Definición de rutas del proyecto

In [46]:
PARQUET_IN = Path("data/processed/familias/VALVULA.parquet")
PARQUET_OUT = Path("data/processed/cleaned/VALVULA.parquet")

PARQUET_OUT.parent.mkdir(parents=True, exist_ok=True)


## Categorización

Limpieza general

In [47]:
def limpiar(txt):
    txt = unicodedata.normalize('NFKD', str(txt)).encode('ascii','ignore').decode()
    txt = re.sub(r'\s+', ' ', txt.upper())
    return txt.strip()


Carga de parquet

In [48]:
df = pd.read_parquet(PARQUET_IN)
print(f'Archivo cargado: {len(df):,} filas')
df['desc_breve'] = df['Descripción Breve'].apply(limpiar)


Archivo cargado: 7,817 filas


Expresiones regullares para extraer información de la descripción

In [49]:
pat_tipo = r'^VAL\s+([A-Z]{3})'
tipos_ok = {'COM','GLO','BOL','MAR','RET','ALI','ESF','AGU','DES'}
pat_diam = r'(\d+\s*-\s*\d+/\d+|\d+\s+\d+/\d+|\d+\s*/\s*\d+|\d+\.\d+|\d+)'
pat_clase = r'(\d{3,4})(?=#|PSI|WOG)'
pat_ext = r'\b(RF|SWXROS|INSXROS|SW|ROS|INS|LUG|BRI)\b'
pat_mat = r'\b(ALC|AC|BR|AI|LAT)\b'

Funciones Auxiliares

In [50]:
def dec_a_frac(num_str):
    f = Fraction(float(num_str)).limit_denominator(16)
    ent, rem = divmod(f.numerator, f.denominator)
    if rem == 0:
        return str(ent)
    frac = f'{rem}/{f.denominator}'
    return f'{ent}-{frac}' if ent else frac

def normalizar_diam(texto):
    m = re.search(pat_diam, texto)
    if not m:
        return None
    tok = m.group(0).strip()
    if re.match(r'\d+\s+\d+/\d+', tok):
        entero, frac = tok.split()
        tok = f'{entero}-{frac}'
    if '.' in tok:
        tok = dec_a_frac(tok)
    return tok


Extracción final

In [51]:
valv = df[df['tipo_material'].str.contains('VALVULA', case=False, na=False)].copy()

valv['Tipo_Valvula']  = valv['desc_breve'].str.extract(pat_tipo)[0].apply(lambda x: x if x in tipos_ok else 'OTR')
valv['Diametro_in']   = valv['desc_breve'].apply(normalizar_diam)
valv['Clase_Libraje'] = valv['desc_breve'].str.extract(pat_clase)[0]
valv['Tipo_Extremos'] = valv['desc_breve'].str.extract(pat_ext)[0].replace({'RF':'BRI'})

valv.loc[
    (valv['Tipo_Extremos'].isna()) &
    (pd.to_numeric(valv['Clase_Libraje'], errors='coerce') <= 600),
    'Tipo_Extremos'
] = 'BRI'

valv['Material_Base'] = valv['desc_breve'].str.extract(pat_mat)[0]

Check valvulas

In [52]:
# Filtrar filas donde la columna 'Descripción Larga' contenga 'ESF' (insensible a mayúsculas)
mask = valv['Descripcion Larga'].str.contains('ESF', case=False, na=False)
valv_filtrado = valv[mask].copy()

# Mostrar las primeras 1000 filas (equivalente a LIMIT 1000)
valv_filtrado.head(10)


,Código (Value),Descripción Breve,Descripcion Larga,UM,Grupo,Jerarquia,tipo_material,desc_breve,Tipo_Valvula,Diametro_in,Clase_Libraje,Tipo_Extremos,Material_Base
10,10103961,"BRONZE BALL VALVE 3"" (DESCONTINUADO)","VALVULA DE ESFERA DE BRONCE 3"". MAKER NO.: B24...",PZA,VALVULAS,OPER_VALV_BOLA,VALVULA,"BRONZE BALL VALVE 3"" (DESCONTINUADO)",OTR,3,NaN,NaN,NaN
25,10135356,"316 STNL ST BALL VALVE 1"", 1WMY4","VALVULA ESFERICA DE DOS PIEZAS, TAMAÑO 1"". CON...",PZA,VALVULAS,OPER_VALV_BOLA,VALVULA,"316 STNL ST BALL VALVE 1"", 1WMY4",OTR,316,NaN,NaN,NaN
36,10159717,"VAL ESF AC- 2""Ø 150# RF - A350LF2","VALVULA ESFERICA, DE PASO COMPLETO, DE 2"" DE D...",PZA,VALVULAS,PROY_VALV_ESFERICA,VALVULA,"VAL ESF AC- 2"" 150# RF - A350LF2",ESF,2,150,BRI,AC
149,10160043,"VALV CONT T/BOLA 6""Ø 900# RTJ",VALVULA DE CONTROL TIPO BOLA MARCA FISHER MODE...,PZA,VALVULAS,PROY_VALV_CONTROL,VALVULA,"VALV CONT T/BOLA 6"" 900# RTJ",OTR,6,900,NaN,NaN
150,10160044,"VALV CONT T/BOLA 8""Ø 900# RTJ",VALVULA DE CONTROL TIPO BOLA MARCA FISHER MODE...,PZA,VALVULAS,PROY_VALV_CONTROL,VALVULA,"VALV CONT T/BOLA 8"" 900# RTJ",OTR,8,900,NaN,NaN
154,10160069,"VALV ESF 2""Ø150#RF A105 API-6D","VALVULA ESFERICA, DE PASO COMPLETO, DE 2"" DE D...",PZA,VALVULAS,PROY_VALV_ESFERICA,VALVULA,"VALV ESF 2""150#RF A105 API-6D",OTR,2,150,BRI,NaN
155,10160095,"VALV ESF A/C 16""Ø 300# RF BRID A-216","VALVULA ESFERICA, DE PASO COMPLETO, DE 16"" DE ...",PZA,VALVULAS,PROY_VALV_ESFERICA,VALVULA,"VALV ESF A/C 16"" 300# RF BRID A-216",OTR,16,300,BRI,NaN
156,10160099,"VALV ESF A/C 2""Ø 300# RF ATORN A-350","VALVULA ESFERICA, DE PASO COMPLETO, DE 2"" DE D...",PZA,VALVULAS,PROY_VALV_ESFERICA,VALVULA,"VALV ESF A/C 2"" 300# RF ATORN A-350",OTR,2,300,BRI,NaN
157,10160124,"VALV ESF A/C 4""Ø 600# RF BRID ASTM A-216","VALVULA ESFERICA, DE PASO COMPLETO, EXTREMOS B...",PZA,VALVULAS,PROY_VALV_ESFERICA,VALVULA,"VALV ESF A/C 4"" 600# RF BRID ASTM A-216",OTR,4,600,BRI,NaN
158,10160126,"VALV ESF A/C 6""Ø 150# RF BRID A-216","VALVULA ESFERICA, DE PASO COMPLETO, DE 6"" DE D...",PZA,VALVULAS,PROY_VALV_ESFERICA,VALVULA,"VALV ESF A/C 6"" 150# RF BRID A-216",OTR,6,150,BRI,NaN


## Exportación de pipeline

In [53]:
cols = [
    'Código (Value)', 'Descripción Breve', 'Descripcion Larga', 'UM', 'Grupo',
    'Jerarquia', 'tipo_material',
    'Diametro_in', 'Material_Base', 'Tipo_Extremos', 'Clase_Libraje', 'Tipo_Valvula'
]
valv[cols].to_parquet(PARQUET_OUT, index=False)
print(f"Archivo limpio exportado a: {PARQUET_OUT.resolve()}")

Archivo limpio exportado a: C:\Users\felip\Documents\GitHub\proyecto_argos\data\processed\cleaned\VALVULA.parquet
